# scikit-learn을 통한 로지스틱 회귀

Scikit-learn은 모델을 만들거나 그와 관련된 여러 도구를 제공하는 머신러닝 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine


In [ ]:

df, y = load_wine(as_frame=True, return_X_y=True)

# quality를 0과 0 아님으로 구분하는 이진 분류를 진행할거다
y[y == 0] = 0
y[y != 0] = 1

# feature와 y를 합치기
df['quality'] = y
label = df['quality'].value_counts().sort_index()
print(label)
X = df.drop('quality', axis=1).values

# 학습 / 테스트 데이터 나누기
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


## Scikit-learn의 주요 사용 방식

주로 다음 규칙을 통해 작동

1. Instance 선언
2. `.fit()`으로 데이터 학습
3. `.predict()`로 예측(모델) 또는 `.transform()`로 수행

In [ ]:
# 1. 전처리 Instance 선언
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# 2. 학습 데이터를 바탕으로 표준화를 위한 사전 작업(평균, 표준편차 등 통계값 계산) 진행
scaler.fit(X_train)

# 3. 통계값을 바탕으로 표준화 진행
X_train_norm = scaler.transform(X_train)
X_train_norm

In [ ]:
# 2-3. fit_transform을 이용하면 fit과 transform을 동시에 진행 가능
X_train_norm = scaler.fit_transform(X_train)
X_train_norm

In [ ]:
# 테스트 데이터를 학습 데이터를 바탕으로 표준화
X_test_norm = scaler.transform(X_test)
# 표준화 작업을 진행할 때 테스트 데이터의 통계를 포함시켜 표준화를 할 경우
# 학습 데이터에 검증 데이터에 대한 정보가 유출될 수 있기 때문에
# 학습 데이터의 통계를 바탕으로 검증 데이터를 표준화
X_test_norm

## Scikit-learn으로 로지스틱 회귀 모델 학습

In [ ]:
# 1. 로지스틱 회귀 Instance 선언
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)

# 2. `.fit()`으로 학습
clf.fit(X_train, y_train)

# 3. X_test로 예측값 생성
y_pred = clf.predict(X_test)
y_pred

### 혼동행렬로 정답 예측 결과 비교

In [ ]:
# 평가: y_pred와 y_test가 얼마나 일치하는가?
from sklearn.metrics import confusion_matrix, classification_report
# 혼동행렬로 정답 예측 결과 비교
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))




### ROC-AUC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_score = clf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_score)
auc = roc_auc_score(y_test, y_score)

# ROC 곡선 시각화 (matplotlib 사용)
plt.plot(fpr, tpr, label=f'ROC-AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()
